# Heston Calibration

We calibrate Heston to the SPX smile using differential evolution.
Heston is a major step up from Black-Scholes — it produces a proper skew.
But as we'll see, it struggles at short maturities. That's exactly where Rough Bergomi takes over.

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
from data.loader import load_spx_options
from models.black_scholes import compute_smile
from models.heston import heston_smile
from calibration.heston_calibrator import calibrate_heston

In [ ]:
r = 0.05
spot, expiry, T, raw_strikes, raw_prices = load_spx_options(expiry_index=4)
strikes, market_ivols = compute_smile(raw_strikes, raw_prices, spot, T, r)

In [ ]:
# Calibrate — this takes a few minutes
params, rmse = calibrate_heston(spot, strikes, market_ivols, T, r)

print(f"\nCalibrated RMSE: {rmse*100:.4f} vol pts")

In [ ]:
# Recompute smile with calibrated parameters at full path count
heston_ivols = heston_smile(spot, strikes, T, r, **{k: params[k] for k in ['v0','kappa','theta','xi','rho']})

atm_idx  = np.argmin(np.abs(strikes - spot))
atm_vol  = market_ivols[atm_idx]
bs_ivols = np.full(len(strikes), atm_vol)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(strikes, market_ivols * 100, 'o-',  color='steelblue',  linewidth=2.5, label='Market')
ax.plot(strikes, heston_ivols * 100, 's--', color='darkorange',  linewidth=2,   label=f'Heston  (RMSE {rmse*100:.3f} vpts)')
ax.plot(strikes, bs_ivols * 100,     ':',   color='tomato',      linewidth=1.5, label='Black-Scholes (flat)')
ax.axvline(x=spot, color='gray', linestyle=':', alpha=0.5, label='ATM')

ax.set_xlabel('Strike')
ax.set_ylabel('Implied Volatility (%)')
ax.set_title(f'Heston vs Market  |  SPX  |  Expiry {expiry}')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/02_heston_calibration.png', dpi=150)
plt.show()

print("\nCalibrated parameters:")
for k, v in params.items():
    print(f"  {k:>6} = {v:.4f}")

Heston does much better than Black-Scholes — the skew is there.
The remaining misfit (especially on the wings) is a known limitation of the model.
It comes from the fact that Heston variance is driven by a standard Brownian motion, which is too smooth.

Realized volatility on SPX has Hurst exponent H ≈ 0.1 — far rougher than BM (H = 0.5).
That's what Rough Bergomi encodes directly.